##Categorical Data Practice
Python for Data Science Chapter 12.1


In [1]:
import numpy as np
import pandas as pd

In [2]:
# Multiplying an array by an integer
values = pd.Series(['apple', 'orange', 'apple', 'apple'] * 2)

In [3]:
values

,0
0,apple
1,orange
2,apple
3,apple
4,apple
5,orange
6,apple
7,apple


In [4]:
pd.unique(values)

array(['apple', 'orange'], dtype=object)

In [5]:
pd.value_counts(values)

<ipython-input-5-e9d61bad5d62>:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(values)


,count
apple,6
orange,2


In [6]:
##In data warehousing, best practice is to use dimension tables contain distinct values and storing the primary observations as interger keys referencing the dimension table
values = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(['apple', 'orange'])
values

,0
0,0
1,1
2,0
3,0
4,0
5,1
6,0
7,0


In [7]:
dim

,0
0,apple
1,orange


In [8]:
# Use the take method to restore the original Series of strings
dim.take(values)

,0
0,apple
1,orange
0,apple
0,apple
0,apple
1,orange
0,apple
0,apple


In [9]:
#pandas has a special Categorical type for holding data that uses the interger-based categorical representation (encoding)
fruits = ['apple', 'orange', 'apple', 'apple'] * 2
N = len(fruits)
df = pd.DataFrame({'fruit': fruits,
                   'basket_id': np.arange(N),
                   'count': np.random.randint(3, 15, size=N),
                   'weight': np.random.uniform(0, 4, size=N)},
                   columns=['basket_id', 'fruit', 'count', 'weight'])

In [10]:
df

,basket_id,fruit,count,weight
0,0,apple,9,1.357560
1,1,orange,14,3.739345
2,2,apple,5,1.670549
3,3,apple,5,2.152067
4,4,apple,7,3.053329
5,5,orange,14,1.475851
6,6,apple,9,3.915834
7,7,apple,3,1.863198


In [11]:
fruit_cat = df['fruit'].astype('category')
fruit_cat

,fruit
0,apple
1,orange
2,apple
3,apple
4,apple
5,orange
6,apple
7,apple


In [12]:
#The values for fruit_cat are not a Numpy array, but an instance of pandas.Categorical:
c = fruit_cat.values
type(c)

pandas.core.arrays.categorical.Categorical

In [13]:
c.categories

Index(['apple', 'orange'], dtype='object')

In [14]:
c.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

In [15]:
df['fruit'] = df['fruit'].astype('category')

In [16]:
df.fruit

,fruit
0,apple
1,orange
2,apple
3,apple
4,apple
5,orange
6,apple
7,apple


In [17]:
# You can also create pandas.Categorical directly from other types of Python sequences
my_categories = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])
my_categories

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

In [18]:
# If you have obtained cateogrical encoded data from another source, you can use the alternative from_codes constructor:
categories = ['foo', 'bar', 'baz']
codes = [0, 1, 2, 0, 0, 1]
my_cats_2 = pd.Categorical.from_codes(codes, categories)
my_cats_2

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

In [19]:
# You can indicate that the categories have a meaningful ordering:
ordered_cat = pd.Categorical.from_codes(codes, categories, ordered=True)
ordered_cat


['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

In [20]:
my_cats_2.as_ordered()

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

#####Computations with Categoricals

In [21]:
# Some parts of pandas, like the groupby function, perform better when working with categoricals.  There are some functions that can utilize the ordered flag
# Create some random numeric data:
np.random.seed(12345)
draws = np.random.randn(1000)
draws[:5]

array([-0.20470766,  0.47894334, -0.51943872, -0.5557303 ,  1.96578057])

In [22]:
# Compute a quartile binning of this data and extract from statistics:
bins = pd.qcut(draws, 4)
bins

[(-0.684, -0.0101], (-0.0101, 0.63], (-0.684, -0.0101], (-0.684, -0.0101], (0.63, 3.928], ..., (-0.0101, 0.63], (-0.684, -0.0101], (-2.9499999999999997, -0.684], (-0.0101, 0.63], (0.63, 3.928]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.9499999999999997, -0.684] < (-0.684, -0.0101] < (-0.0101, 0.63] <
                                           (0.63, 3.928]]

In [23]:
# While useful, the exact sample quartiles may be less useful for producing a report than quartile names.  We can achieve this with the labels argument to qcut:
bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
bins.codes[:10]

array([1, 2, 1, 1, 3, 3, 2, 2, 3, 3], dtype=int8)

In [24]:
# The labeled bins categorical does not contain information about the bin edges in the data, so we can use groupby to extract some summary statistics
bins = pd.Series(bins, name='quartile')
results = (pd.Series(draws)
          .groupby(bins)
          .agg(['count', 'min', 'max'])
          .reset_index())
results

<ipython-input-24-4a1c37d4526c>:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(bins)


,quartile,count,min,max
0,Q1,250,-2.949343,-0.685484
1,Q2,250,-0.683066,-0.010115
2,Q3,250,-0.010032,0.628894
3,Q4,250,0.634238,3.927528


In [25]:
# The quartile column in the result retains the original categorical information, including ordering from bins:
results['quartile']

,quartile
0,Q1
1,Q2
2,Q3
3,Q4


In [26]:
# Better performance with categoricals: A categorical version of a DataFrame column will often use significantly less memory.
N = 10000000
draws =pd.Series(np.random.randn(N))
labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))
categories = labels.astype('category')
labels.memory_usage()

80000128

In [27]:
categories.memory_usage()

10000332

#####Categorical Methods

In [28]:
s = pd.Series(['a' ,'b', 'c', 'd'] * 2)
cat_s = s.astype('category')
cat_s

,0
0,a
1,b
2,c
3,d
4,a
5,b
6,c
7,d


In [30]:
# The special cat attribute provides access to categorical methods:
cat_s.cat.codes

,0
0,0
1,1
2,2
3,3
4,0
5,1
6,2
7,3


In [31]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

In [33]:
# We can use the set_categories method to change them:
actual_categories = ['a', 'b', 'c', 'd', 'e']
cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2

,0
0,a
1,b
2,c
3,d
4,a
5,b
6,c
7,d


In [34]:
# The new categories will be reflected in operations that use them:
cat_s.value_counts()

,count
a,2
b,2
c,2
d,2


In [36]:
cat_s2.value_counts()

,count
a,2
b,2
c,2
d,2
e,0


In [37]:
# use remove_unused_categories method to trim unobserved categories
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]
cat_s3

,0
0,a
1,b
4,a
5,b


In [39]:
cat_s3.cat.remove_unused_categories()

,0
0,a
1,b
4,a
5,b


In [40]:
# Use one-hot encoding to transform categorical data into dummy variables:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2, dtype='category')
pd.get_dummies(cat_s)

,a,b,c,d
0,True,False,False,False
1,False,True,False,False
2,False,False,True,False
3,False,False,False,True
4,True,False,False,False
5,False,True,False,False
6,False,False,True,False
7,False,False,False,True
